In [ ]:
#This is pyenv file for making AI classification.

In [2]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()

/Users/rkuge/.venvs/lede/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [32]:
data=pd.read_csv('data2025.csv')

In [33]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [34]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
6103,307189,25V744000,LEXUS,LX,2025,25TB13/25LB06/WRE25,BACK OVER PREVENTION:DISPLAY FUNCTION,Toyota Motor Engineering & Manufacturing,20210730.0,20250807.0,...,Toyota Motor Engineering & Manufacturing (Toyo...,A rearview camera that fails to display an ima...,Dealers will update the parking assist softwar...,NaN,260709005636652000001883,"Computer, Parking Assist",Parking Assist ECU,86792-33521,No,No
3503,298236,25V595000,TOYOTA,RAV4,2025,See Part 573 Report,ELECTRICAL SYSTEM: INSTRUMENT CLUSTER/PANEL,Toyota Motor Engineering & Manufacturing,NaN,20250228.0,...,Toyota Motor Engineering & Manufacturing (Toyo...,An instrument panel display that does not show...,Dealers will update the instrument panel softw...,NaN,260511005636692000001084,"Meter Assy, Combination",Combination Meter,83800-3AT11,No,No
4859,303507,25V065000,HYUNDAI,IONIQ 5,2025,275,"SERVICE BRAKES, HYDRAULIC:POWER ASSIST:ELECTRI...",Hyundai Motor America,20231218.0,20241210.0,...,Hyundai Motor America (Hyundai) is recalling c...,Reduced braking performance can increase the d...,Owners who have not received the recall remedy...,Owners may also contact the National Highway T...,127912005648760000001665,Integrated Electrical Booster,IEB Integrated Electrical Booster (IONIQ 5 N s...,58520-NI000,No,No
7552,313837,25V566000,NEW FLYER,XE40,2025,NaN,ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION B...,"New Flyer of America, Inc.",NaN,20250701.0,...,"New Flyer of America, Inc. (New Flyer) is reca...",A battery module that overheats increases the ...,New Flyer will inspect and replace the battery...,NaN,260478005669706000001305,HV module,BATTERY-XALT XMOD111E,765786,No,No
6215,307441,25V744000,TOYOTA,CROWN SIGNIA,2025,25TB13/25LB06/WRE25,BACK OVER PREVENTION:DISPLAY FUNCTION,Toyota Motor Engineering & Manufacturing,20231113.0,20250807.0,...,Toyota Motor Engineering & Manufacturing (Toyo...,A rearview camera that fails to display an ima...,Dealers will update the parking assist softwar...,NaN,260709005650989000001883,"Computer, Parking Assist",Parking Assist ECU,86792-48A81,No,No


In [35]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

In [45]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [37]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [39]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-nano):  gpt-5.4-mini


In [40]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [41]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $1.29'

In [42]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

In [51]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify)

100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:39<00:00,  1.26it/s]


In [44]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
6103,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2022-2026 Toyota, Lexus, and Subaru Solterra vehicles equipped with a Panoramic View Monitor (PVM) system. Please see the recall report for a complete list of models. A software error may cause the rearview camera to freeze or display a blank screen when the vehicle is in reverse. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMVSS) number 111, ""Rear Visibility.""",software
3503,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2023-2024 Toyota Venza, 2023-2025 RAV4 Prime, RAV4, Highlander, GR Corolla, Crown, 2024-2025 Lexus TX, LS, Toyota Tacoma, Grand Highlander, and 2025 Lexus RX, Toyota Crown Signia, Camry, RAV 4 Plug-in Hybrid (PHEV), and 4 Runner vehicles. Due to an error in the instrument panel software at vehicle startup, the instrument panel may fail to display vehicle speed, brake system, and tire pressure warning lights.",software
4859,"Hyundai Motor America (Hyundai) is recalling certain 2025 Hyundai IONIQ 5 ""N"" vehicles equipped with a left-foot braking (LFB) feature. The integrated electronic brake (IEB) and vehicle control unit (VCU) software may cause reduced braking performance when LFB is activated.",software
7552,"New Flyer of America, Inc. (New Flyer) is recalling certain 2013, 2017-2025 XE40, 2019-2025 XE60, XE35, 2023-2025 XHE40, and 2024-2025 XHE60 buses. An inadequate weld on the high voltage battery busbar and incorrect software may cause the battery module to overheat.",software
6215,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2022-2026 Toyota, Lexus, and Subaru Solterra vehicles equipped with a Panoramic View Monitor (PVM) system. Please see the recall report for a complete list of models. A software error may cause the rearview camera to freeze or display a blank screen when the vehicle is in reverse. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMVSS) number 111, ""Rear Visibility.""",software
1049,"Winnebago Industries, Inc. (Winnebago) is recalling certain 2014-2018 Sightseer, Itasca Sunova, 2016-2025 Forza, Vista, and Itasca/Winnebago Sunstar, 2015-2025 Adventurer, 2015-2024 Commercial Vehicle, 2015-2028 Itasca Suncruiser, and 2016-2017 Itasca Solei recreational vehicles. The hood bracket adhesive may have been incorrectly applied which can allow the bracket to detach.",others
7049,"Ford Motor Company (Ford) is recalling certain 2025 Lincoln Navigator, F-150, Expedition, Bronco, and Ranger vehicles. The Electronic Brake Booster (EBB) module may malfunction while driving or when the driver is utilizing an Advanced Driver Assistance System (ADAS) feature, resulting in a loss of power brake assist.",electronics
5028,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2022-2026 Toyota, Lexus, and Subaru Solterra vehicles equipped with a Panoramic View Monitor (PVM) system. Please see the recall report for a complete list of models. A software error may cause the rearview camera to freeze or display a blank screen when the vehicle is in reverse. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMVSS) number 111, ""Rear Visibility.""",software
7190,"Autocar, LLC (Autocar) is recalling certain 2023-2026 ACX (Xpeditor) and DC (Legend) vehicles. The steering wheel nut may be improperly secured, allowing the steering wheel to separate from the steering column.",chasies
1310,"PACCAR Incorporated (PACCAR) is recalling certain 2021-2025 Peterbilt and Kenworth vehicles, equipped with Bendix EC80 Advanced Electronic Control Units (ECU). Please see the recall report for a complete list of models. Electrical noise and low signal to the power line carrier may cause the ECU to incorrectly process commands or stop functioning.",electronics


In [47]:
okay_to_do_the_whole_thing = input()

In [48]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [49]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify)
data.to_csv(data_dir + 'Recall_classified2025.csv')

100%|██████████████████████████████████████████████████████████████████████████████| 9617/9617 [1:56:40<00:00,  1.37it/s]


In [50]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-fa3...4da",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 17.450273,
    "include_byok_in_limit": false,
    "usage": 29.64878875,
    "usage_daily": 2.549727,
    "usage_weekly": 23.9902673,
    "usage_monthly": 27.72183485,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


In [52]:
import sys
print(sys.executable)      # /Users/rkuge/.venvs/lede/bin/python と一致するか
from tqdm.auto import tqdm
print(tqdm)                # ウィジェット版かテキスト版か
from ipywidgets import IntSlider
IntSlider()                # そのノートブックでスライダーが出るか

/Users/rkuge/.venvs/lede/bin/python
<class 'tqdm.asyncio.tqdm_asyncio'>


IntSlider(value=0)